In [290]:
import os
import pandas as pd

In [291]:
raw_data_path = os.path.join(os.path.dirname(os.getcwd()), "data", "raw", "MCDI_Overview_10-04-2025.xlsx")

df = pd.read_excel(raw_data_path, sheet_name="Parametric Analysis", engine="openpyxl", skiprows=2)

c:\Users\david\AppData\Local\pypoetry\Cache\virtualenvs\capbioco2-optimization-LdjI1cg_-py3.13\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)
c:\Users\david\AppData\Local\pypoetry\Cache\virtualenvs\capbioco2-optimization-LdjI1cg_-py3.13\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Conditional Formatting extension is not supported and will be removed
  warn(msg)


In [292]:
df.rename(columns={df.columns[0]: "Series", df.columns[1]: "Measurement"}, inplace=True)
df["Series"] = df["Series"].ffill()

def move_column(df, col_name, new_position):
    cols = list(df.columns)
    cols.insert(new_position, cols.pop(cols.index(col_name)))
    return df[cols]

def drop_columns_from(df, col_start):
    idx = list(df.columns).index(col_start)
    return df.iloc[:, :idx]

df = move_column(df, "Check", 2)
df = drop_columns_from(df, "Test Remarks ")


In [293]:
# Drop all measurements that are not valid
df = df[df["Check"] == "Valid"]

# Drop all measurements with Mol.\n(g/L) unequal to 0.5
df = df[df["Mol.\n(g/L)"] == 0.5]

# Drop all measurements with H2O unequal to DI
df = df[df["H2O\n"] == "DI"]

# Drop the 4th measurement series
df = df[df["Series"] != "4 -  pH Series"]

# Drop all measurements with a time in seconds unequal to 155/45/120
df = df[df["Time\n(s)"] == "155/45/120"]

In [294]:
# Prepare input data

df_input = df[["Series", "Measurement", "Flow\n (L/min)", "Curr. Lim.\n(A)"]].copy()

df_input[["Flow_C", "Flow_PP", "Flow_P"]] = df_input["Flow\n (L/min)"].str.split("/", expand=True)
df_input.drop(columns="Flow\n (L/min)", inplace=True)

df_input["Current_P"] = df_input["Curr. Lim.\n(A)"].str.split("/", expand=True)[2]
df_input.drop(columns="Curr. Lim.\n(A)", inplace=True)

columns_to_convert_input = ["Flow_C", "Flow_PP", "Flow_P", "Current_P"]
for col in columns_to_convert_input:
    df_input[col] = df_input[col].str.replace(',', '.', regex=False).astype(float)

In [295]:
# Prepate output data

df_output = df[["Series", "Measurement", "Remov. \n(%)", "Ry \n(%)"]].copy()
df_output.rename(columns={"Remov. \n(%)": "Removal", "Ry \n(%)": "Recovery"}, inplace=True)

columns_to_convert_output = ["Removal", "Recovery"]
for col in columns_to_convert_output:
    df_output[col] = df_output[col].astype(float)

In [296]:
def normalize(df, bounds):
    df_norm = df.copy()
    for col in bounds:
        a, b = bounds[col]
        df_norm[col] = (df[col] - a) / (b - a)
    return df_norm

def denormalize(norm_values, bounds):
    return {
        k: norm_values[k] * (bounds[k][1] - bounds[k][0]) + bounds[k][0]
        for k in norm_values
    }

input_bounds = {
    'Flow_C': (0.2, 1.5),
    'Flow_PP': (0.2, 1.5),
    'Flow_P': (0.2, 1.5),
    'Current_P': (35, 55)
}

output_bounds = {
    'Removal': (0, 100),
    'Recovery': (0, 100),
}

df_input_norm = normalize(df_input, input_bounds)
df_output_norm = normalize(df_output, output_bounds)


In [ ]:
processed_data_path = os.path.join(os.path.dirname(os.getcwd()), "data", "processed")
input_data_path = os.path.join(os.path.dirname(os.getcwd()), "data", "processed", "InputData.csv")
output_data_path = os.path.join(os.path.dirname(os.getcwd()), "data", "processed", "OutputData.csv")

df_input_norm.to_csv(input_data_path, index=False)
df_output_norm.to_csv(output_data_path, index=False)

In [298]:
df_read_back_input = pd.read_csv(input_data_path)
df_read_back_input

,Series,Measurement,Flow_C,Flow_PP,Flow_P,Current_P
0,"1 - Pre-Optimization 0,5",1.10,0.192308,0.230769,0.230769,1.0
1,"1 - Pre-Optimization 0,5",1.20,0.115385,0.153846,0.153846,1.0
2,"1 - Pre-Optimization 0,5",1.30,0.076923,0.076923,0.076923,1.0
3,"1 - Pre-Optimization 0,5",1.40,0.038462,0.038462,0.038462,1.0
4,"1 - Pre-Optimization 0,5",1.50,0.269231,0.307692,0.307692,1.0
5,2 - Current/Flow-Field 0.5 g/l,2.20,0.192308,0.230769,0.230769,1.0
6,2 - Current/Flow-Field 0.5 g/l,2.30,0.230769,0.269231,0.269231,1.0
7,2 - Current/Flow-Field 0.5 g/l,2.40,0.269231,0.307692,0.307692,1.0
8,2 - Current/Flow-Field 0.5 g/l,2.50,0.307692,0.346154,0.346154,1.0
9,2 - Current/Flow-Field 0.5 g/l,2.60,0.346154,0.384615,0.384615,1.0


In [ ]:
df_read_back_output = pd.read_csv(output_data_path)
df_read_back_output